# Project 21 — Hybrid Retrieval Lab
## Compare BM25, Dense, and Hybrid Retrieval Locally

**What you'll learn:**
- Implement BM25 (sparse) retrieval
- Implement dense vector retrieval
- Combine both into a hybrid retriever with Reciprocal Rank Fusion
- Measure and compare retrieval quality

**Stack:** Ollama · LangChain · ChromaDB · rank-bm25 · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb rank-bm25

## Step 1 — Setup and Sample Corpus

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

corpus = [
    Document(page_content="Python is a high-level programming language known for readability. "
        "It supports multiple paradigms including OOP and functional programming.",
        metadata={"id": 1, "topic": "python"}),
    Document(page_content="Rust provides memory safety without garbage collection through "
        "its ownership system. It competes with C++ for systems programming.",
        metadata={"id": 2, "topic": "rust"}),
    Document(page_content="Machine learning models learn patterns from data. Supervised "
        "learning uses labeled examples while unsupervised finds hidden structures.",
        metadata={"id": 3, "topic": "ml"}),
    Document(page_content="Vector databases store embeddings for similarity search. "
        "Popular options include ChromaDB, FAISS, Pinecone, and Weaviate.",
        metadata={"id": 4, "topic": "vectordb"}),
    Document(page_content="RAG combines retrieval with generation. The retriever finds "
        "relevant documents and the generator produces answers grounded in evidence.",
        metadata={"id": 5, "topic": "rag"}),
    Document(page_content="BM25 is a bag-of-words retrieval function that ranks documents "
        "based on term frequency and inverse document frequency with saturation.",
        metadata={"id": 6, "topic": "retrieval"}),
    Document(page_content="Transformers use self-attention to process sequences in parallel. "
        "The architecture consists of encoder and decoder stacks with multi-head attention.",
        metadata={"id": 7, "topic": "transformers"}),
    Document(page_content="Fine-tuning adapts a pre-trained model to specific tasks using "
        "smaller domain-specific datasets. LoRA reduces the parameters needed.",
        metadata={"id": 8, "topic": "finetuning"}),
]
print(f"Corpus: {len(corpus)} documents")

## Step 2 — BM25 Sparse Retriever

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize for BM25
tokenized_corpus = [doc.page_content.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k = np.argsort(scores)[::-1][:k]
    results = []
    for idx in top_k:
        results.append({"doc": corpus[idx], "score": float(scores[idx]), "rank": len(results)+1})
    return results

print("BM25 results for 'vector database similarity search':")
for r in bm25_search("vector database similarity search"):
    print(f"  [{r['rank']}] score={r['score']:.3f} — {r['doc'].page_content[:60]}...")

## Step 3 — Dense Vector Retriever

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(corpus, embeddings, collection_name="hybrid_lab")
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def dense_search(query, k=3):
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [{"doc": doc, "score": float(score), "rank": i+1}
            for i, (doc, score) in enumerate(results)]

print("Dense results for 'vector database similarity search':")
for r in dense_search("vector database similarity search"):
    print(f"  [{r['rank']}] score={r['score']:.3f} — {r['doc'].page_content[:60]}...")

## Step 4 — Hybrid Retrieval with Reciprocal Rank Fusion

In [ ]:
def reciprocal_rank_fusion(bm25_results, dense_results, k=60):
    """Combine BM25 and dense results using RRF."""
    fused_scores = {}

    for result_list in [bm25_results, dense_results]:
        for r in result_list:
            doc_id = r["doc"].metadata["id"]
            rank = r["rank"]
            if doc_id not in fused_scores:
                fused_scores[doc_id] = {"doc": r["doc"], "score": 0, "sources": []}
            fused_scores[doc_id]["score"] += 1.0 / (k + rank)
            fused_scores[doc_id]["sources"].append(
                f"{'BM25' if result_list == bm25_results else 'Dense'} rank={rank}"
            )

    sorted_results = sorted(fused_scores.values(), key=lambda x: x["score"], reverse=True)
    return sorted_results

def hybrid_search(query, k=3):
    bm25_r = bm25_search(query, k=5)
    dense_r = dense_search(query, k=5)
    fused = reciprocal_rank_fusion(bm25_r, dense_r)
    return fused[:k]

print("Hybrid (RRF) results for 'vector database similarity search':")
for i, r in enumerate(hybrid_search("vector database similarity search")):
    print(f"  [{i+1}] rrf_score={r['score']:.4f} — {r['doc'].page_content[:60]}...")
    print(f"       Sources: {r['sources']}")

## Step 5 — Comparative Evaluation

In [ ]:
test_queries = [
    ("What is BM25?", [6]),  # Expected: doc about BM25
    ("How does RAG work?", [5]),  # Expected: RAG doc
    ("memory safe systems language", [2]),  # Expected: Rust doc
    ("neural network architecture", [7]),  # Expected: Transformers doc
]

print(f"{'Query':<35} {'BM25':>6} {'Dense':>6} {'Hybrid':>6}")
print("-" * 60)

for query, expected_ids in test_queries:
    bm25_top = bm25_search(query, k=1)[0]["doc"].metadata["id"]
    dense_top = dense_search(query, k=1)[0]["doc"].metadata["id"]
    hybrid_top = hybrid_search(query, k=1)[0]["doc"].metadata["id"]

    def check(result_id):
        return "✓" if result_id in expected_ids else "✗"

    print(f"{query:<35} {check(bm25_top):>6} {check(dense_top):>6} {check(hybrid_top):>6}")

## What You Learned
- **BM25 sparse retrieval** — keyword-based, good for exact matches
- **Dense vector retrieval** — semantic, good for paraphrases
- **Reciprocal Rank Fusion** — combines both for robust results
- **Comparative evaluation** with ground truth